In [1]:
#pip install fastkml

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import geopandas as gpd
from shapely.geometry import Point
import glob
import matplotlib.pyplot as plt
from scipy.signal import periodogram
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go


In [7]:
event_rate_threshold

0.34626121635094714

In [8]:
import pandas as pd
import numpy as np

# Load proton density data
proton_data = pd.read_csv("CELIAS_Proton_Monitor_Hourly.csv") 
proton_data['datetime'] = pd.to_datetime(proton_data['datetime'])
proton_data.set_index('datetime', inplace=True)

# Load earthquake data
event_df = pd.read_csv("declustered_earthquake_dataset_usgs.csv")
event_df['time'] = pd.to_datetime(event_df['time'], format='mixed')

# Check if the datetime column is timezone-aware
if event_df['time'].dt.tz is None:
    # Localize to 'UTC' if not already timezone-aware
    event_df['time'] = event_df['time'].dt.tz_localize('UTC')
    
event_df['date'] = event_df['time'].dt.date
event_df = event_df[(event_df['mag'] > 5.6) & (event_df['depth'] < 60)]

# Ensure proton_data index is timezone-aware if needed
if proton_data.index.tz is None:
    proton_data.index = proton_data.index.tz_localize('UTC')

# Calculate total number of days
total_hours = proton_data.shape[0]
total_days = (proton_data.index[-1] - proton_data.index[0]).days + 1
print(f"Total days in dataset: {total_days}")

# Calculate total number of days and total number of events
total_days = (proton_data.index[-1] - proton_data.index[0]).days + 1
total_events = event_df.shape[0]

# Initialize lists to store results
best_V_T = None
best_fraction_of_failure = float('inf')

# Calculate event rate threshold
event_rate_threshold = event_df.shape[0] / total_days


def calculate_event_relative_rate(EC, DC, total_events, total_days):
    if DC == 0 or (total_events - EC) == 0:
        return 0  # Avoid division by zero
    R = (EC / DC) / ((total_events - EC) / (total_days - DC))
    return R

# Loop through different V_T values in range 15-31
for V_T in np.arange(15, 32, 1):
    print(f"Testing V_T: {V_T}")

    # Add a column to check if Np is above the threshold
    proton_data['above_threshold'] = proton_data['Np'] > V_T

    # Count the number of days satisfying the condition DC
    satisfying_days = proton_data[proton_data['above_threshold']].resample('D').size()
    DC = satisfying_days[satisfying_days > 0].count()  # Number of days with density above threshold

    # Get all the dates where the condition is satisfied
    satisfying_dates = satisfying_days[satisfying_days > 0].index.date
    satisfying_dates_set = set(satisfying_dates)

    # Count the number of events EC on those days
    EC = event_df[event_df['date'].isin(satisfying_dates_set)].shape[0]

    # Calculate Event Relative Rate
    R = calculate_event_relative_rate(EC, DC, total_events, total_days)

    # Determine if prediction is a failure
    fraction_of_failure = R <= (total_events / total_days)

    # Update best V_T based on fraction of failure
    if not fraction_of_failure:
        if R < best_fraction_of_failure:
            best_fraction_of_failure = R
            best_V_T = V_T

    print(f"V_T: {V_T}")
    print(f"DC: {DC}, EC: {EC}")
    print(f"Event Relative Rate R: {R}")
    print(f"Fraction of failures: {fraction_of_failure}")

# Print summary of results
print(f"Best V_T: {best_V_T}")
print(f"Fraction of failures for best V_T: {best_fraction_of_failure}")

Total days in dataset: 10030
Testing V_T: 15
V_T: 15
DC: 2580, EC: 1479
Event Relative Rate R: 0.9950502828409805
Fraction of failures: False
Testing V_T: 16
V_T: 16
DC: 2318, EC: 1333
Event Relative Rate R: 0.9993012733001247
Fraction of failures: False
Testing V_T: 17
V_T: 17
DC: 2075, EC: 1199
Event Relative Rate R: 1.00539111827889
Fraction of failures: False
Testing V_T: 18
V_T: 18
DC: 1859, EC: 1082
Event Relative Rate R: 1.014244937764796
Fraction of failures: False
Testing V_T: 19
V_T: 19
DC: 1657, EC: 977
Event Relative Rate R: 1.0298052861792388
Fraction of failures: False
Testing V_T: 20
V_T: 20
DC: 1495, EC: 876
Event Relative Rate R: 1.0216772968116397
Fraction of failures: False
Testing V_T: 21
V_T: 21
DC: 1349, EC: 786
Event Relative Rate R: 1.0146475006933326
Fraction of failures: False
Testing V_T: 22
V_T: 22
DC: 1197, EC: 688
Event Relative Rate R: 0.9988089115831745
Fraction of failures: False
Testing V_T: 23
V_T: 23
DC: 1078, EC: 617
Event Relative Rate R: 0.9941274

In [2]:
import pandas as pd
import numpy as np

# Load proton density data
proton_data = pd.read_csv("CELIAS_Proton_Monitor_Hourly.csv")  # Update path as needed
proton_data['datetime'] = pd.to_datetime(proton_data['datetime'])
proton_data.set_index('datetime', inplace=True)

# Load earthquake data
event_df = pd.read_csv("declustered_earthquake_dataset_usgs.csv")
event_df['time'] = pd.to_datetime(event_df['time'], format='mixed')

# Check if the datetime column is timezone-aware
if event_df['time'].dt.tz is None:
    # Localize to 'UTC' if not already timezone-aware
    event_df['time'] = event_df['time'].dt.tz_localize('UTC')
    
event_df['date'] = event_df['time'].dt.date
event_df = event_df[(event_df['mag'] > 5.6) & (event_df['depth'] < 60)]

# Calculate total number of days and total number of events
total_days = (proton_data.index[-1] - proton_data.index[0]).days + 1
total_events = event_df.shape[0]

# Compute V values
V_av = np.mean(proton_data['Np'])
V_min = np.min(proton_data['Np'])
V_max = np.max(proton_data['Np'])

# Function to calculate event relative rate
def calculate_event_relative_rate(EC, DC, total_events, total_days):
    if DC == 0 or (total_events - EC) == 0:
        return 0  # Avoid division by zero
    R = (EC / DC) / ((total_events - EC) / (total_days - DC))
    return R

# Generate synthetic datasets and compute R values
num_synthetic = 10**5
synthetic_R_values = []

for _ in range(num_synthetic):
    # Create synthetic dataset
    synthetic_events = event_df.copy()
    synthetic_events['time'] = np.random.permutation(synthetic_events['time'].values)
    synthetic_events['date'] = synthetic_events['time'].dt.date
    
# Calculate EC and DC for synthetic data
for V_T in np.arange(15, 31, 1):
    proton_data['above_threshold'] = proton_data['Np'] > V_T
    satisfying_days = proton_data[proton_data['above_threshold']].resample('D').size()
    DC = satisfying_days[satisfying_days > 0].count()
    satisfying_dates = satisfying_days[satisfying_days > 0].index.date
    satisfying_dates_set = set(satisfying_dates)
    EC = synthetic_events[synthetic_events['date'].isin(satisfying_dates_set)].shape[0]
    R = calculate_event_relative_rate(EC, DC, total_events, total_days)
    synthetic_R_values.append(R)

# Compute R for real data
real_R_values = []
for V_T in np.arange(15, 32, 1):
    proton_data['above_threshold'] = proton_data['Np'] > V_T
    satisfying_days = proton_data[proton_data['above_threshold']].resample('D').size()
    DC = satisfying_days[satisfying_days > 0].count()
    satisfying_dates = satisfying_days[satisfying_days > 0].index.date
    satisfying_dates_set = set(satisfying_dates)
    EC = event_df[event_df['date'].isin(satisfying_dates_set)].shape[0]
    R = calculate_event_relative_rate(EC, DC, total_events, total_days)
    real_R_values.append((V_T, R))

# Compare real R values to synthetic R values
for V_T, real_R in real_R_values:
    significant = np.all(np.array(synthetic_R_values) < real_R)
    print(f"V_T: {V_T}, Real R: {real_R}")
    if significant:
        print(f"Real R is significantly higher than all synthetic R values for V_T: {V_T}")
    else:
        print(f"Real R is not significantly higher for V_T: {V_T}")

V_T: 15, Real R: 1.0161103919420884
Real R is not significantly higher for V_T: 15
V_T: 16, Real R: 1.0103407980888568
Real R is not significantly higher for V_T: 16
V_T: 17, Real R: 1.0185104850890443
Real R is not significantly higher for V_T: 17
V_T: 18, Real R: 1.0139576119241387
Real R is not significantly higher for V_T: 18
V_T: 19, Real R: 1.0425655323977365
Real R is not significantly higher for V_T: 19
V_T: 20, Real R: 1.0327113272281285
Real R is not significantly higher for V_T: 20
V_T: 21, Real R: 1.0245528755414928
Real R is not significantly higher for V_T: 21
V_T: 22, Real R: 1.0344560789147306
Real R is not significantly higher for V_T: 22
V_T: 23, Real R: 1.0232618979615837
Real R is not significantly higher for V_T: 23
V_T: 24, Real R: 1.0305960256121327
Real R is not significantly higher for V_T: 24
V_T: 25, Real R: 1.0487366529231419
Real R is not significantly higher for V_T: 25
V_T: 26, Real R: 1.0951751565037164
Real R is not significantly higher for V_T: 26
V_T:

In [ ]:
# # Find the index of the maximum Event Relative Rate
# best_index = np.argmax(event_relative_rates_array)

# # Get the best prediction values
# best_V_step = V_steps_array[best_index]
# best_V_T = V_T_values_array[best_index]
# best_first_drop_day_counts_array = first_drop_day_counts_array[best_index]
# best_earthquake_counts_array = earthquake_counts_array[best_index]
# best_event_relative_rate = event_relative_rates_array[best_index]

# print(f"Best V_step: {best_V_step}")
# print(f"Best V_T: {best_V_T}")
# print(f"Best Event Relative Rate: {best_event_relative_rate}")

In [ ]:
# best_first_drop_day_counts_array

In [ ]:
# best_earthquake_counts_array